# Mutual Fund Recommender — LangChain edition (Colab runner)

All the logic lives in the `backend/mf_recommender` package; this notebook installs it, finds your files and serves the API for the React app.

**Pipeline:** goal → (Phi-2 query rewrite) → hybrid retrieval (FAISS + BM25, fused with reciprocal rank fusion, filtered by category & risk) → peer-relative ranking → live NAV → grounded explanations (Phi-2 + LoRA, with a fact-checked fallback).

**Setup (no paths to edit):**
1. Unzip the project on your computer and upload the `mutual_funds_recommendation_system` folder anywhere in My Drive (e.g. `My Drive/Colab_Notebooks/`).
2. Runtime → Change runtime type → **T4 GPU**.
3. Store your ngrok token in Colab Secrets (🔑 icon) as `NGROK_AUTHTOKEN` (only needed for the API step).
4. Runtime → Run all.

Everything the notebook creates (search index, retrained adapter) is saved in the project's `artifacts/` folder in Drive, so it's reused next time.

In [ ]:
import os, sys, glob
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'

def find_in_drive(relative_path, max_depth=4):
    """Paths in My Drive ending in `relative_path`, up to `max_depth` folders deep, newest first."""
    hits = []
    for depth in range(0, max_depth + 1):
        hits += glob.glob(os.path.join(DRIVE, *(['*'] * depth), relative_path))
    return sorted(set(hits), key=os.path.getmtime, reverse=True)

# 1) The project folder: wherever you put it, whatever the folders are called.
MARKER = os.path.join('backend', 'mf_recommender', 'service.py')
projects = [h[: -len(MARKER) - 1] for h in find_in_drive(MARKER)]
if not projects:
    raise FileNotFoundError('Project not found in My Drive: upload the unzipped mutual_funds_recommendation_system folder')
PROJECT_DIR = projects[0]  # newest upload wins
if len(projects) > 1:
    print('Several copies found; using the newest. Older copies (safe to delete):', projects[1:])
print('Project:', PROJECT_DIR)

BACKEND_DIR = os.path.join(PROJECT_DIR, 'backend')
os.chdir(BACKEND_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)  # so `import mf_recommender` works in this notebook

# 2) Catch an incomplete upload early.
EXPECTED = ['backend/mf_recommender/retrieval.py', 'backend/tests/conftest.py',
            'backend/scripts/train_lora.py', 'backend/pytest.ini']
missing = [f for f in EXPECTED if not os.path.exists(os.path.join(PROJECT_DIR, f))]
if missing:
    print('WARNING: missing from your Drive copy:', missing, '- re-upload the full folder from the zip')

# 3) The fund data: the project's data/ folder, otherwise the newest copy anywhere in My Drive.
DATA_CSV = os.path.join(PROJECT_DIR, 'data', 'updated_data_with_nav.csv')
if not os.path.isfile(DATA_CSV):
    elsewhere = find_in_drive('updated_data_with_nav.csv')
    if not elsewhere:
        raise FileNotFoundError('updated_data_with_nav.csv not found; upload the data/ folder from the zip')
    DATA_CSV = elsewhere[0]
    os.environ['MF_DATA_PATH'] = DATA_CSV
print('Fund data:', DATA_CSV)

In [ ]:
# Installs everything directly (doesn't depend on the requirements files).
!pip install -q "langchain-core>=1.0,<2" "langchain-community>=0.4,<0.5" "langchain-huggingface>=1.0,<2" \
    faiss-cpu rank-bm25 sentence-transformers fastapi "uvicorn[standard]" \
    transformers peft accelerate datasets pyngrok pytest httpx

## Configure
Set `MF_LLM_BACKEND = 'none'` for a fast, template-only run without loading Phi-2. Other settings: `backend/.env.example`.

In [ ]:
import json
from mf_recommender.config import ARTIFACTS_DIR, find_adapter

os.environ['MF_LLM_BACKEND'] = 'hf'
os.environ['MF_BASE_MODEL'] = 'microsoft/phi-2'
os.environ['MF_LIVE_NAV'] = 'true'

def phi2_adapters_in_drive():
    """LoRA adapters for Phi-2 anywhere in My Drive (e.g. the original notebook's fine_tuned_phi2)."""
    out = []
    for cfg in find_in_drive('adapter_config.json'):
        if 'checkpoint' in cfg:
            continue
        try:
            base = json.load(open(cfg)).get('base_model_name_or_path', '')
        except Exception:
            continue
        if 'phi-2' in base.lower():
            out.append(os.path.dirname(cfg))
    return out

# Adapter: the project's artifacts/ (e.g. after retraining below), else any Phi-2 adapter in your Drive.
adapter = find_adapter() or next(iter(phi2_adapters_in_drive()), None)
if adapter:
    os.environ['MF_ADAPTER_PATH'] = adapter
    print('Using LoRA adapter:', adapter)
else:
    os.environ.pop('MF_ADAPTER_PATH', None)
    print('No LoRA adapter found; using base Phi-2 (works; retrain below for better explanations)')
print('Index and artifacts folder:', ARTIFACTS_DIR)

## Check and build the index
Tests run offline (no model downloads) in a few seconds.

In [ ]:
!python -m pytest -q   # expect: 71 passed

In [ ]:
!python -m mf_recommender.build_index

## Try it in Python

In [ ]:
import pandas as pd
from mf_recommender.config import Settings
from mf_recommender.service import RecommenderService

service = RecommenderService.from_settings(Settings.from_env())  # loads embeddings + Phi-2 once

result = service.recommend(
    'I want long-term growth over 10 years with low costs',
    category=None,   # None = let the goal decide; or e.g. 'Large Cap Mutual Funds'
    risk='High',
)
print('Rewritten query:', result.query['rewritten'])
print('Notice:', result.notice)
display(pd.DataFrame(result.funds)[['scheme_name', 'category', 'risk_level', 'NAV', 'returns_1yr',
                                    'returns_3yr', 'returns_5yr', 'expense_ratio', 'score']])
for f in result.funds:
    print(f"\n{f['scheme_name']} [{f['explanation_source']}]\n  {f['explanation']}")

## (Optional) Retrain the LoRA adapter
`scripts/train_lora.py` fixes the original training: loss only on the answer, an EOS the model actually learns (so it stops instead of rambling), dynamic padding, data-grounded targets and a held-out evaluation. Roughly 30–60 minutes on a T4. The adapter is saved to `artifacts/fine_tuned_phi2_v2` and picked up automatically next time (re-run the Configure cell).

In [ ]:
RETRAIN = False
if RETRAIN:
    !python scripts/train_lora.py --gradient-checkpointing

## Serve the API through ngrok
Reuses the `service` loaded above, so Phi-2 isn't loaded twice. Put the printed URL in `frontend/.env` as `REACT_APP_API_URL`.

In [ ]:
import threading
import uvicorn
from google.colab import userdata
from pyngrok import conf, ngrok
from mf_recommender.api import create_app

app = create_app(service=service)
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info'))
threading.Thread(target=server.run, daemon=True).start()

conf.get_default().auth_token = userdata.get('NGROK_AUTHTOKEN')
public_url = ngrok.connect(8000).public_url
print('REACT_APP_API_URL =', public_url)

In [ ]:
import requests
r = requests.post(f'{public_url}/recommend',
                  json={'goal': 'Park my emergency fund safely for 3 months', 'category': None, 'risk': 'Low'},
                  headers={'ngrok-skip-browser-warning': 'true'}, timeout=300)
r.raise_for_status()
[(f['scheme_name'], f['category']) for f in r.json()['funds']]

In [ ]:
# Stop the server and tunnel
server.should_exit = True
ngrok.kill()